# Question 1 — Stage 5 and Stage 6: Feature engineering and finalization

**Group number:** 4  
**Student 1:** Niyonsenga Felicien  
**Student 2:** Leonce  
**Question:** Q1 — Maize yield in Rwanda  
**Stage:** 5–6

Create features only from information expected to be available before harvest. The target `yield_t_ha` is not used to calculate any predictor. The identifier `record_id` is not a model feature.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
sns.set_theme()
CWD = Path.cwd()
raw_candidates = [
    CWD / "data" / "raw" / "maize_yield_rwanda_rw.csv",
    CWD / "q1_maize_yield" / "data" / "raw" / "maize_yield_rwanda_rw.csv",
    CWD.parent / "data" / "raw" / "maize_yield_rwanda_rw.csv",
]
RAW_PATH = next((p for p in raw_candidates if p.exists()), None)
if RAW_PATH is None:
    raise FileNotFoundError("Could not find data/raw/maize_yield_rwanda_rw.csv. Open the notebook from the extracted project.")
PROJECT = RAW_PATH.parents[2]
INTERIM = PROJECT / "data" / "interim"
PROCESSED = PROJECT / "data" / "processed"
INTERIM.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(RAW_PATH)

from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

## Create at least five engineered features
1. `total_fertiliser_kg_ha`: sum of NPK and urea application rates.
2. `total_organic_inorganic_input`: sum of manure, NPK and urea (units differ; treat as a simple combined indicator cautiously and explain the limitation).
3. `rainfall_per_degree`: rainfall divided by average temperature.
4. `altitude_band`: categorical grouping of altitude.
5. `soil_ph_group`: acidic / near-neutral grouping.
6. `fertiliser_balance_npk_urea`: ratio of NPK to urea with a small denominator safeguard.
7. `season_year`: season and year combination.

In [2]:
feat = df.drop_duplicates().copy()
feat["total_fertiliser_kg_ha"] = feat["npk_kg_ha"] + feat["urea_kg_ha"]
feat["total_input_indicator"] = feat["npk_kg_ha"] + feat["urea_kg_ha"] + feat["manure_t_ha"]
feat["rainfall_per_degree"] = feat["rainfall_mm"] / feat["avg_temp_c"].replace(0, np.nan)
feat["altitude_band"] = pd.cut(feat["altitude_m"], bins=[-np.inf,1500,1800,2100,np.inf],
                               labels=["Low","Mid-low","Mid-high","High"])
feat["soil_ph_group"] = pd.cut(feat["soil_ph"], bins=[-np.inf,5.5,6.5,np.inf],
                               labels=["Acidic","Slightly acidic to near neutral","Above 6.5"])
feat["npk_urea_ratio"] = feat["npk_kg_ha"] / feat["urea_kg_ha"].replace(0, np.nan)
feat["season_year"] = feat["year"].astype(str) + "_" + feat["season"].astype(str)
display(feat[["total_fertiliser_kg_ha","total_input_indicator","rainfall_per_degree",
              "altitude_band","soil_ph_group","npk_urea_ratio","season_year","yield_t_ha"]].head())

,total_fertiliser_kg_ha,total_input_indicator,rainfall_per_degree,altitude_band,soil_ph_group,npk_urea_ratio,season_year,yield_t_ha
0,175.2,176.61,NaN,Low,NaN,2.052265,2015_A,2.71
1,100.8,104.49,22.162896,Low,Slightly acidic to near neutral,0.607656,2015_A,4.09
2,171.8,176.98,44.331288,Mid-high,Slightly acidic to near neutral,2.388560,2015_A,2.58
3,154.1,155.78,48.439024,Mid-high,Acidic,1.551325,2015_A,2.45
4,154.5,154.86,35.202186,Mid-high,Slightly acidic to near neutral,1.194602,2015_A,2.09


## Feature Justification Table
Review the units and usefulness of each feature with your partner. In particular, `total_input_indicator` combines kg/ha and tonnes/ha, so it is only a rough indicator and should not be interpreted as a physical total. You may choose to remove it from the final model feature set.

In [3]:
feature_justification = pd.DataFrame([
    ("total_fertiliser_kg_ha","npk_kg_ha + urea_kg_ha","Summarises inorganic fertiliser application; relevant to input/yield question."),
    ("rainfall_per_degree","rainfall_mm / avg_temp_c","Combines two climate measures into a simple climate indicator."),
    ("altitude_band","Bins altitude_m into four bands","Makes geographic elevation groups easier to compare."),
    ("soil_ph_group","Groups soil pH into three bands","Represents soil acidity categories."),
    ("npk_urea_ratio","npk_kg_ha / urea_kg_ha","Describes balance between two fertiliser application rates."),
    ("season_year","year + season","Captures the time period and agricultural season."),
    ("total_input_indicator","NPK + urea + manure","Exploratory combined indicator only; units are mixed, so use cautiously.")
], columns=["feature","creation","justification"])
display(feature_justification)
feature_justification.to_csv(PROJECT/"docs"/"feature_justification.csv", index=False)

,feature,creation,justification
0,total_fertiliser_kg_ha,npk_kg_ha + urea_kg_ha,Summarises inorganic fertiliser application; r...
1,rainfall_per_degree,rainfall_mm / avg_temp_c,Combines two climate measures into a simple cl...
2,altitude_band,Bins altitude_m into four bands,Makes geographic elevation groups easier to co...
3,soil_ph_group,Groups soil pH into three bands,Represents soil acidity categories.
4,npk_urea_ratio,npk_kg_ha / urea_kg_ha,Describes balance between two fertiliser appli...
5,season_year,year + season,Captures the time period and agricultural season.
6,total_input_indicator,NPK + urea + manure,Exploratory combined indicator only; units are...


## Correlation with target (exploratory)
This is an association check, not model training and not proof of causation. Use only numerical columns for the correlation table.

In [4]:
numeric_feature_view = feat.select_dtypes(include=np.number)
corr_with_target = numeric_feature_view.corr(numeric_only=True)["yield_t_ha"].sort_values(ascending=False)
display(corr_with_target.to_frame("correlation_with_yield"))

,correlation_with_yield
yield_t_ha,1.000000
rainfall_mm,0.363716
rainfall_per_degree,0.309281
plant_density_per_ha,0.295246
total_input_indicator,0.254666
total_fertiliser_kg_ha,0.252033
urea_kg_ha,0.233992
npk_kg_ha,0.222850
manure_t_ha,0.137967
year,0.134116


## Final feature selection demonstration
For a transparent baseline, remove the identifier and target from predictors, create the engineered variables, then use a training-only imputation/scaling and SelectKBest score. The selector is fitted on training data only. Do not use the validation or test data to choose features.

In [5]:
TARGET="yield_t_ha"
ID="record_id"
model_feat = feat.dropna(subset=[TARGET]).copy()
X = model_feat.drop(columns=[TARGET, ID])
y = model_feat[TARGET].astype(float)
X = pd.get_dummies(X, drop_first=False)
X = X.replace([np.inf,-np.inf], np.nan)
X_train, X_temp, y_train, y_temp = train_test_split(X,y,test_size=.30,random_state=42)
X_valid, X_test, y_valid, y_test = train_test_split(X_temp,y_temp,test_size=.50,random_state=42)
imputer=SimpleImputer(strategy="median")
scaler=StandardScaler()
Xtr=imputer.fit_transform(X_train)
Xva=imputer.transform(X_valid)
Xte=imputer.transform(X_test)
Xtr=scaler.fit_transform(Xtr)
Xva=scaler.transform(Xva)
Xte=scaler.transform(Xte)
k=min(15, Xtr.shape[1])
selector=SelectKBest(score_func=f_regression,k=k)
Xtr_sel=selector.fit_transform(Xtr,y_train)
Xva_sel=selector.transform(Xva)
Xte_sel=selector.transform(Xte)
selected_names=X_train.columns[selector.get_support()].tolist()
print("Selected features:", selected_names)
print("Selected shapes:", Xtr_sel.shape, Xva_sel.shape, Xte_sel.shape)

Selected features: ['rainfall_mm', 'npk_kg_ha', 'urea_kg_ha', 'plant_density_per_ha', 'pest_incidence_index', 'total_fertiliser_kg_ha', 'total_input_indicator', 'rainfall_per_degree', 'season_A', 'season_C', 'seed_variety_Local', 'seed_variety_PAN691', 'seed_variety_RHM104', 'farming_practice_Improved', 'farming_practice_Traditional']
Selected shapes: (998, 15) (214, 15) (214, 15)


## Save final engineered splits

In [6]:
def frame_with_target(arr, index, target):
    out=pd.DataFrame(arr, columns=selected_names, index=index)
    out[TARGET]=target
    return out
frame_with_target(Xtr_sel,X_train.index,y_train).to_csv(PROCESSED/"train_engineered_selected.csv",index=False)
frame_with_target(Xva_sel,X_valid.index,y_valid).to_csv(PROCESSED/"validation_engineered_selected.csv",index=False)
frame_with_target(Xte_sel,X_test.index,y_test).to_csv(PROCESSED/"test_engineered_selected.csv",index=False)
feat.to_csv(PROCESSED/"maize_engineered_reference.csv",index=False)
print("Final files saved in:", PROCESSED.resolve())

Final files saved in: C:\Users\LENOVO\Documents\btech\machine learning\Machine_Learning_Ready_Datasets\ML_Maize_Yield_JupyterLab_Project\ML_Maize_Yield_JupyterLab_Project\q1_maize_yield\data\processed


## Data Card — draft
- **Purpose:** prepare maize farm-season records for later yield prediction.
- **Target:** `yield_t_ha` (tonnes per hectare).
- **Source:** CSV supplied with the coursework; detailed collection metadata was not included.
- **Size:** 1,426 rows and 20 original columns (verify using notebook output).
- **Main missing fields:** rainfall, soil pH and pest incidence index.
- **Preprocessing:** duplicate check, text trimming, median/mode imputation and scaling/encoding fitted on training data.
- **Feature engineering:** fertiliser total, rainfall-temperature ratio, altitude band, soil pH group, NPK/urea ratio and season-year.
- **Limitations:** confirm dataset provenance, units, synthetic/real status, index scale and suitability of random splitting for forecasting.
- **Reproduction:** install dependencies and run notebooks 01 to 04 in order.